# AeroPulse Anomaly — 04: Hierarchical Baseline (Experiments A1–A2)

Replace the flat `location_id × hour_of_week` baseline with one that degrades
gracefully to nearby stations, then to region, then to nation.
Implements plan sections 5 and 23.

## The problem this fixes

Notebook 03's baseline is keyed on `location_id`. That works perfectly for a
station present in training and fails completely for one that is not:

```
temporal split   -> 0%   fallback   (every station appears in both slices)
spatial holdout  -> 100% fallback   (a new station has no key at all)
```

100% fallback means every held-out station gets a single national hour-of-week
median. AeroPulse's roadmap points at 1-km grid cells and unmonitored
locations, and a baseline that cannot answer for an unseen point cannot serve
any of them.

## The hierarchy

```
                 query (location, hour_of_week)
                              |
              +---------------+----------------+
              |    1. station's own history    |   needs training rows here
              +---------------+----------------+
                              | absent
              +---------------+----------------+
              |    2. inverse-distance blend   |   <= 150 km, up to 8 stations
              |       of nearby trained sites  |   weight 1/d^2
              +---------------+----------------+
                              | absent
              +---------------+----------------+
              |    3. regional climatology     |   IGP flag x 2-degree block
              +---------------+----------------+
                              | absent
              +---------------+----------------+
              |    4. national hour-of-week    |
              +--------------------------------+
```

Level 2 is the whole point. Air quality is spatially autocorrelated at the
100 km scale, so a station 40 km away is a far better prior than a national
average — and the plan's `w1·station + w2·nearby + w3·regional` formulation is
implemented here as a strict precedence chain, which is easier to audit than
fitted blend weights and needs no extra training data.

### Experiments

| ID | Baseline |
|---|---|
| A1 | flat `location_id × hour_of_week` (notebook 03's) |
| A2 | hierarchical, strict precedence |
| A2b | hierarchical with distance shrinkage |

A2b exists because strict precedence trusts a neighbour 300 km away exactly as
much as one 5 km away. It shrinks the neighbour estimate toward the regional
climatology as the nearest contributing station recedes:

```
w = 1 / (1 + (d_nearest / 25 km)^2)
expected = w * neighbour_IDW + (1 - w) * regional
```

In [1]:
# ============================================================================
# CONFIGURATION
# Every anomaly notebook shares this block so paths, thresholds and helper
# definitions cannot drift between phases. The real logic lives in
# anomaly_toolkit.py — the "Shared Feature Layer" of the improvement plan.
# ============================================================================
from pathlib import Path
import os, json, sys, warnings, time

import numpy as np
import pandas as pd
from dotenv import find_dotenv, load_dotenv

_dotenv_path = find_dotenv(usecwd=True)
if not _dotenv_path:
    raise RuntimeError(
        "No .env found. Copy this folder's .env.example to .env, or launch "
        "Jupyter from a directory whose parent has .env."
    )
load_dotenv(_dotenv_path, override=True)
warnings.filterwarnings("ignore")

sys.path.insert(0, str(Path.cwd()))
import anomaly_toolkit as tk

CFG = tk.load_config(os.getenv("AEROPULSE_PROJECT_ROOT", "."))
FEATURE_FILE = CFG["feature_dir"] / "anomaly_features.parquet"
PM25_BASE = CFG["pm25_processed"] / "base"

print("env      :", _dotenv_path)
print("project  :", CFG["project_root"])
print("pm25 data:", CFG["pm25_data_root"])
print("hazard   : PM2.5 >=", CFG["very_poor_ugm3"], "ug/m3 within", CFG["horizon_hours"], "h")

env      : /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/anomaly_detector/.env
project  : /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/anomaly_detector
pm25 data: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator
hazard   : PM2.5 >= 121.0 ug/m3 within 24 h


In [2]:
# ============================================================================
# LOAD
# ============================================================================
manifest = json.loads((CFG["feature_dir"] / "dataset_manifest.json").read_text())
COLS = ["timestamp_utc", "location_id", "lat", "lon", "pm25", "hour_of_week", "season"]
df = pd.read_parquet(FEATURE_FILE, columns=COLS)
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)
df = df.sort_values(["location_id", "timestamp_utc"]).reset_index(drop=True)

geometry = pd.read_parquet(CFG["feature_dir"] / "station_geometry.parquet")
print(f"{len(df):,} rows / {df['location_id'].nunique()} stations")
print(f"geometry: {len(geometry)} stations, {geometry['region_key'].nunique()} regions, "
      f"{geometry['geo_block'].nunique()} geo blocks")

1,705,252 rows / 149 stations
geometry: 149 stations, 50 regions, 48 geo blocks


## 1. Two holdout designs

A **temporal** split answers "does the baseline hold up next month?" A
**spatial** split answers "does it hold up at a station we have never
measured?" The plan is clear that these are different questions and that only
the second one exposes the flat baseline's failure.

In [3]:
# ============================================================================
# SPLITS
# ============================================================================
q70 = df["timestamp_utc"].quantile(0.70)
temporal_train = df[df["timestamp_utc"] < q70]
temporal_test = df[df["timestamp_utc"] >= q70]

rng = np.random.default_rng(42)
ids = np.array(sorted(df["location_id"].unique()))
held_stations = set(rng.permutation(ids)[:30].tolist())
spatial_train = df[~df["location_id"].isin(held_stations)]
spatial_test = df[df["location_id"].isin(held_stations)]

print(f"temporal: train {len(temporal_train):,} / test {len(temporal_test):,}")
print(f"spatial : train {len(spatial_train):,} ({len(ids)-len(held_stations)} stations) "
      f"/ test {len(spatial_test):,} ({len(held_stations)} stations)")

temporal: train 1,193,612 / test 511,640
spatial : train 1,356,785 (119 stations) / test 348,467 (30 stations)


In [4]:
# ============================================================================
# A1 — FLAT BASELINE (the notebook 03 incumbent)
# ============================================================================
def fit_flat(frame):
    agg = (frame.groupby(["location_id", "hour_of_week"])["pm25"]
                .agg(baseline_pm25="median", n="count").reset_index())
    return agg[agg["n"] >= 5].drop(columns="n")

def apply_flat(table, frame, national_median):
    out = frame[["location_id", "hour_of_week"]].merge(
        table, on=["location_id", "hour_of_week"], how="left")
    level = np.where(out["baseline_pm25"].notna(), "station", "national")
    return out["baseline_pm25"].fillna(national_median).to_numpy("float32"), level

results = []
for split, tr, te in (("temporal", temporal_train, temporal_test),
                      ("spatial", spatial_train, spatial_test)):
    tbl = fit_flat(tr)
    nat = float(tr["pm25"].median())
    pred, level = apply_flat(tbl, te, nat)
    m = tk.regression_metrics(te["pm25"], pred)
    results.append({"experiment": "A1 flat", "split": split, **m,
                    "station_share": float((level == "station").mean()),
                    "national_share": float((level != "station").mean())})
    print(f"A1 {split:<9} MAE {m['mae']:.2f}  R2 {m['r2']:.3f}  "
          f"fallback {(level!='station').mean():.1%}")

A1 temporal  MAE 15.40  R2 -0.040  fallback 0.0%


A1 spatial   MAE 22.55  R2 -0.076  fallback 100.0%


In [5]:
# ============================================================================
# A2 — HIERARCHICAL BASELINE
# ============================================================================
VARIANTS = {"A2 precedence": None, "A2b shrunk": 25.0}

for label, shrink in VARIANTS.items():
    for split, tr, te in (("temporal", temporal_train, temporal_test),
                          ("spatial", spatial_train, spatial_test)):
        t0 = time.time()
        hb = tk.HierarchicalBaseline(neighbour_radius_km=150.0, max_neighbours=8,
                                     shrink_half_distance_km=shrink).fit(tr, geometry)
        out = hb.transform(te)
        m = tk.regression_metrics(te["pm25"], out["baseline_pm25"])
        shares = out["baseline_level"].value_counts(normalize=True)
        results.append({"experiment": label, "split": split, **m,
                        "station_share": float(shares.get("station", 0.0)),
                        "national_share": float(shares.get("national", 0.0)
                                                + shares.get("global", 0.0))})
        print(f"{label:<15} {split:<9} MAE {m['mae']:.2f}  R2 {m['r2']:.3f}  ({time.time()-t0:.0f}s)")
        print("   resolution:", {k: f"{v:.1%}" for k, v in shares.items() if v > 0.001})
        if label == "A2b shrunk":
            globals()[f"levels_{split}"] = out

A2 precedence   temporal  MAE 15.40  R2 -0.040  (3s)
   resolution: {'station': '100.0%'}


A2 precedence   spatial   MAE 22.13  R2 -0.047  (2s)
   resolution: {'neighbour': '96.7%', 'national': '3.3%'}


A2b shrunk      temporal  MAE 15.40  R2 -0.040  (3s)
   resolution: {'station': '100.0%'}


A2b shrunk      spatial   MAE 22.12  R2 -0.043  (2s)
   resolution: {'neighbour_shrunk': '60.1%', 'neighbour': '36.6%', 'national': '3.3%'}


In [6]:
# ============================================================================
# COMPARISON
# ============================================================================
comparison = pd.DataFrame(results)[
    ["experiment", "split", "n", "mae", "rmse", "r2", "bias", "station_share", "national_share"]]
print(comparison.round(3).to_string(index=False))

piv = comparison.pivot_table(index="split", columns="experiment", values="mae")
for c in ("A2 precedence", "A2b shrunk"):
    piv[f"{c} vs A1 %"] = (100 * (piv["A1 flat"] - piv[c]) / piv["A1 flat"]).round(2)
print()
print("MAE by split (positive % = better than flat):")
print(piv.round(3).to_string())

   experiment    split      n    mae   rmse     r2    bias  station_share  national_share
      A1 flat temporal 511640 15.401 25.892 -0.040   6.129            1.0           0.000
      A1 flat  spatial 348467 22.552 41.806 -0.076 -11.118            0.0           1.000
A2 precedence temporal 511640 15.401 25.892 -0.040   6.129            1.0           0.000
A2 precedence  spatial 348467 22.129 41.227 -0.047 -11.869            0.0           0.033
   A2b shrunk temporal 511640 15.401 25.892 -0.040   6.129            1.0           0.000
   A2b shrunk  spatial 348467 22.123 41.154 -0.043 -11.703            0.0           0.033

MAE by split (positive % = better than flat):
experiment  A1 flat  A2 precedence  A2b shrunk  A2 precedence vs A1 %  A2b shrunk vs A1 %
split                                                                                    
spatial      22.552         22.129      22.123                   1.88                 1.9
temporal     15.401         15.401      15.401       

### Result — the hierarchy works structurally, but pays little numerically

Three things to read here, and only the first is the one the plan predicted.

**1. Temporal parity is correct.** Every station has training history, so A2
resolves at level 1 and *is* A1. A hierarchy that changed this number would be
a bug.

**2. The structural gap closes.** A1 has no answer at all for a held-out
station — 100% of rows collapse to one national curve. A2 resolves ~97% of
them at the neighbour level from trained stations within 150 km. AeroPulse can
now be asked about an unmonitored location and return something location-aware.

**3. The accuracy gain is small — about 2% MAE.** This is worth stating
plainly rather than dressing up. The plan calls a hierarchical baseline
"essential" for spatial generalisation, and structurally it is; but the
measured error reduction on unseen stations is marginal.

The reason is visible in the R² column: **both baselines have R² near zero.**
An hour-of-week climatology captures the diurnal and weekly cycle and nothing
else — day-to-day variance in Indian PM2.5 is driven by weather, fire and
transport, none of which a climatology sees. Improving a component that
explains ~0% of the variance cannot move much.

The actionable conclusion: **spatial generalisation will not be won in the
baseline layer.** It has to come from the model, which does have `lat`/`lon`,
elevation, regional PM2.5 and meteorology to work with. Notebook 07 tests that
directly rather than assuming the baseline fix was sufficient.

## 2. Where does the fallback actually land?

An aggregate fallback share hides whether the hierarchy degrades sensibly. If
a held-out station in the Indo-Gangetic Plain resolves against a neighbour
300 km away in a different pollution regime, level 2 is doing harm.

In [7]:
# ============================================================================
# RESOLUTION QUALITY ON HELD-OUT STATIONS
# ============================================================================
lv = levels_spatial.copy()
lv["location_id"] = spatial_test["location_id"].to_numpy()
lv["pm25"] = spatial_test["pm25"].to_numpy()
lv["abs_err"] = (lv["pm25"] - lv["baseline_pm25"]).abs()

by_level = lv.groupby("baseline_level", observed=True).agg(
    rows=("pm25", "size"), mae=("abs_err", "mean"), mean_pm25=("pm25", "mean")).dropna()
print("held-out station error by resolution level:")
print(by_level.round(2).to_string())

geo_idx = geometry.set_index("location_id")
held_df = geo_idx.loc[sorted(held_stations)]
trained_ids = [i for i in geometry["location_id"] if i not in held_stations]
trained_geo = geo_idx.loc[trained_ids]
nn = []
for loc, row in held_df.iterrows():
    d = tk.haversine_km(row["lat"], row["lon"],
                        trained_geo["lat"].to_numpy(), trained_geo["lon"].to_numpy())
    nn.append({"location_id": loc, "nearest_trained_km": float(d.min()),
               "n_within_150km": int((d <= 150).sum()),
               "igp": int(row.get("is_indo_gangetic_plain", 0))})
nn = pd.DataFrame(nn)
print(f"\nheld-out station -> nearest trained station distance:")
print(nn["nearest_trained_km"].describe().round(1).to_string())
print(f"held-out stations with no trained neighbour inside 150 km: "
      f"{int((nn['n_within_150km'] == 0).sum())} / {len(nn)}")

held-out station error by resolution level:
                    rows        mae  mean_pm25
baseline_level                                
neighbour         127444  17.809999  31.959999
neighbour_shrunk  209393  25.240000  44.580002
national           11630  13.240000  22.090000

held-out station -> nearest trained station distance:
count     30.0
mean      66.8
std       66.2
min        1.8
25%        7.6
50%       55.5
75%       97.6
max      304.7
held-out stations with no trained neighbour inside 150 km: 1 / 30


### Result — read this table carefully, it is confounded

The instinct is to check that MAE rises as the resolution level drops. It does
not, and that is **not** evidence the hierarchy is broken.

Compare the `mean_pm25` column alongside `mae`. Rows resolved at the national
level belong to *cleaner* stations, which have lower absolute PM2.5 and
therefore lower absolute error. The two groups are different subpopulations,
so their MAEs are not comparable and neither ordering would prove anything.

What the table does establish is where each level is used and how much traffic
it carries. The nearest-trained-neighbour distances are the more informative
diagnostic: the median held-out station has a trained station within ~55 km,
but the distribution has a long tail out past 300 km, and one station has no
trained neighbour inside 150 km at all.

That tail is exactly what motivates the shrinkage variant. Under strict
precedence a 300 km neighbour is trusted as fully as a 5 km one — and the
block holdout below shows that assumption doing real damage.

## 3. Geographic block holdout

Plan section 23 warns that a *random* station holdout is optimistic: a
held-out Delhi station probably still has three trained Delhi stations nearby,
sharing the same pollution regime. Holding out whole 2° blocks removes that
crutch and is the harder, more honest test.

In [8]:
# ============================================================================
# GEOGRAPHIC BLOCK HOLDOUT
# ============================================================================
blocks = geometry.groupby("geo_block")["location_id"].apply(list)
blocks = blocks[blocks.apply(len) >= 2]
rng = np.random.default_rng(7)
held_blocks = rng.permutation(blocks.index.to_numpy())[:max(1, len(blocks) // 5)]
held_by_block = {s for b in held_blocks for s in blocks[b]}

blk_train = df[~df["location_id"].isin(held_by_block)]
blk_test = df[df["location_id"].isin(held_by_block)]
print(f"holding out {len(held_blocks)} blocks / {len(held_by_block)} stations "
      f"/ {len(blk_test):,} rows")

nat_med = float(blk_train["pm25"].median())
flat_tbl = fit_flat(blk_train)
flat_pred, flat_level = apply_flat(flat_tbl, blk_test, nat_med)
block_rows = [{"experiment": "A1 flat",
               **tk.regression_metrics(blk_test["pm25"], flat_pred),
               "station_share": 0.0,
               "national_share": float((flat_level != "station").mean())}]
for label, shrink in VARIANTS.items():
    hbb = tk.HierarchicalBaseline(shrink_half_distance_km=shrink).fit(blk_train, geometry)
    hier = hbb.transform(blk_test)
    sh = hier["baseline_level"].value_counts(normalize=True)
    block_rows.append({"experiment": label,
                       **tk.regression_metrics(blk_test["pm25"], hier["baseline_pm25"]),
                       "station_share": float(sh.get("station", 0.0)),
                       "national_share": float(sh.get("national", 0.0) + sh.get("global", 0.0))})
    print(f"{label:<15} resolution:", {k: round(v, 3) for k, v in sh.items() if v > 0.001})

block_cmp = pd.DataFrame(block_rows)
print()
print(block_cmp[["experiment", "n", "mae", "rmse", "r2", "national_share"]].round(3).to_string(index=False))

holding out 6 blocks / 18 stations / 205,379 rows


A2 precedence   resolution: {'neighbour': 0.844, 'national': 0.156}


A2b shrunk      resolution: {'neighbour_shrunk': 0.844, 'national': 0.156}

   experiment      n    mae   rmse     r2  national_share
      A1 flat 205379 19.622 32.387 -0.033           1.000
A2 precedence 205379 20.409 33.161 -0.083           0.156
   A2b shrunk 205379 19.622 32.338 -0.030           0.156


### Result — the block holdout is where precedence fails

This is the most instructive table in the notebook, because the naive
hierarchy **loses to the flat baseline it was meant to replace**.

Under `A2 precedence`, holding out whole 2° blocks leaves only distant
neighbours, and inverse-distance weighting happily interpolates a Delhi-like
station from sites hundreds of kilometres away in an entirely different
pollution regime. The flat baseline's national median, crude as it is, is
*less wrong* than a confidently misplaced local estimate.

`A2b shrunk` repairs this. Weighting the neighbour estimate by
`1 / (1 + (d/25 km)²)` pulls distant interpolations back toward the regional
climatology, restoring at least parity with the flat baseline while keeping
the genuine short-range signal where a close neighbour exists. **A2b is what
gets persisted.**

Two honest consequences:

- A random station holdout is optimistic, precisely as plan section 23 warns.
  It kept the flaw invisible; only the block holdout exposed it.
- With the aggressive shrinkage that makes A2b safe, most of the neighbour
  estimate's weight is transferred to the regional climatology. On a
  149-station network, neighbour interpolation is adding little over a decent
  regional prior. That will change as the network densifies, but claiming a
  large win from it today would not be supported by these numbers.

## 4. Persist

In [9]:
# ============================================================================
# PERSIST
# ============================================================================
import joblib
P4 = CFG["artifact_root"] / "phase4_baseline"
P4.mkdir(parents=True, exist_ok=True)

hb_full = tk.HierarchicalBaseline(shrink_half_distance_km=25.0).fit(temporal_train, geometry)
joblib.dump({"baseline": hb_full, "geometry": geometry,
             "schema_version": CFG["schema_version"],
             "fitted_on": "temporal_train (first 70% by time)"},
            P4 / "hierarchical_baseline.joblib", compress=3)

full_cmp = pd.concat([comparison, block_cmp.assign(split="geo_block")], ignore_index=True)
full_cmp.to_csv(P4 / "baseline_comparison.csv", index=False)

tk.write_report(P4 / "phase4_report.json", {
    "experiments": {"A1": "flat location_id x hour_of_week",
                    "A2": "hierarchical, strict precedence",
                    "A2b": "hierarchical with 25 km distance shrinkage (shipped)"},
    "generated_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "comparison": full_cmp.to_dict("records"),
    "held_stations_random": sorted(int(x) for x in held_stations),
    "held_blocks": [str(b) for b in held_blocks],
    "neighbour_radius_km": 150.0, "max_neighbours": 8,
    "resolution_shares_spatial": levels_spatial["baseline_level"].value_counts(
        normalize=True).round(4).to_dict(),
    "notes": [
        "Temporal parity between A1 and A2 is expected — every station has history.",
        "A2's value is entirely on unseen stations and unseen geographic blocks.",
        "baseline_level is a confidence signal for the alert engine in notebook 08.",
        "Low absolute R2 is by design: an hour-of-week climatology defines "
        "'normal', it does not forecast.",
    ],
})
print("wrote", P4 / "phase4_report.json")

wrote /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/anomaly_detector/artifacts/anomaly/phase4_baseline/phase4_report.json


## Conclusion

**Delivered.** The flat baseline is replaced by A2b. An unseen station now
resolves through distance-shrunk interpolation from trained neighbours instead
of a single national curve, and `baseline_level` / `baseline_neighbour_km` come
out as per-row confidence signals the alert engine in notebook 08 can consume.

**But the plan overstates what this buys.** Section 23 presents the
hierarchical baseline as the fix for spatial generalisation. Measured:

- ~2% MAE improvement on random station holdout.
- Parity, not improvement, on the harder geographic-block holdout — and the
  naive version was actively *worse* there until shrinkage was added.
- Both baselines sit at R² ≈ 0, because an hour-of-week climatology cannot
  explain day-to-day variance driven by weather, fire and transport.

**Revised recommendation.** Treat the hierarchical baseline as a correctness
fix — it removes a structural dead end and stops the detector claiming a
national average is a local expectation — not as an accuracy lever. Spatial
generalisation has to be won in the model layer, where `lat`/`lon`, elevation,
regional PM2.5 and meteorology are available. Notebook 07 tests that claim
instead of inheriting it.

One design note: the plan's `w1·station + w2·nearby + w3·regional` formulation
is implemented as precedence plus a *distance-derived* shrinkage weight rather
than fitted weights. Fitting three blend weights against a single 30-station
holdout would mostly fit those 30 stations. The shrinkage form has one
interpretable parameter, and the block holdout above is what set it.